# PySpark semi join



# Initalise a spark session

In [1]:
# Initalise a spark session
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.functions import col

# Fix JAVA_HOME to your actual Java 21 path
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages io.delta:delta-spark_2.12:3.2.0 pyspark-shell"

# Build Spark session with Delta Lake support
builder = SparkSession.builder \
    .appName("SemiJoin") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()


26/05/11 22:42:27 WARN Utils: Your hostname, DESKTOP-OQT8U26 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/11 22:42:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/robyip/projects/pyspark-deltalake/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/robyip/.ivy2/cache
The jars for the packages stored in: /home/robyip/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-533b993d-518d-43d0-a23e-65442e36cb73;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 189ms :: artifacts dl 9ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0 

# Create 2 dataframe of test data

In [7]:
employees = spark.createDataFrame([
    (1, "Alice", 10),
    (2, "Bob",   20),
    (3, "Carol", 30),
    (4, "Dave",  10),
], ["emp_id", "name", "dept_id"])

departments = spark.createDataFrame([
    (10, "Engineering"),
    (20, "Marketing"),
    (20, "e-Marketing"),
    # dept 30 is missing
], ["dept_id", "dept_name"])

# Semi join between the 2 dataframes


In [11]:
# left_semi: employees whose dept exists in departments
# Returns the left hand table if there is a match 
semi_result = employees.join(departments, on="dept_id", how="left_semi")
semi_result.show()

# inner join: employees whose dept exists in departments
# Showing how this would look very differently if we used inner join
semi_result = employees.join(departments, on="dept_id", how="inner")
semi_result.show()


+-------+------+-----+
|dept_id|emp_id| name|
+-------+------+-----+
|     10|     1|Alice|
|     10|     4| Dave|
|     20|     2|  Bob|
+-------+------+-----+

+-------+------+-----+-----------+
|dept_id|emp_id| name|  dept_name|
+-------+------+-----+-----------+
|     10|     1|Alice|Engineering|
|     10|     4| Dave|Engineering|
|     20|     2|  Bob|  Marketing|
|     20|     2|  Bob|e-Marketing|
+-------+------+-----+-----------+



# Left anti join - show left table row where there isn't to the right table

In [12]:
# left_anti: employees whose dept does NOT exist
anti_result = employees.join(departments, on="dept_id", how="left_anti")
anti_result.show()


+-------+------+-----+
|dept_id|emp_id| name|
+-------+------+-----+
|     30|     3|Carol|
+-------+------+-----+

